# Seed Inventra (J&F)

# IMPORTS E CONFIGURAÇÃO INICIAL

In [36]:
from faker import Faker
import psycopg2
from psycopg2.extras import execute_values
from dotenv import load_dotenv
import itertools
import random
import re
import unicodedata
import os
from datetime import timedelta

fake = Faker("pt_BR")
load_dotenv()
random.seed(42)
Faker.seed(42)

print("Bibliotecas carregadas")

Bibliotecas carregadas


# CONEXÃO COM O BANCO DE DADOS

In [37]:
conn = psycopg2.connect(
    host=os.getenv("DB_HOST"),
    port=os.getenv("DB_PORT"),
    database=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    sslmode=os.getenv("DB_SSLMODE", "require")
)
cursor = conn.cursor()

print("Conectado ao PostgreSQL")

Conectado ao PostgreSQL


# RESET TOTAL DO BANCO

In [38]:
cursor.execute("""
TRUNCATE TABLE
    tb_profile,
    tb_kitchen,
    tb_category,
    tb_measurement_unit,
    tb_user,
    tb_product,
    tb_supplier,
    tb_product_supplier,
    tb_product_kitchen_parameter,
    tb_stock_batch,
    tb_requisition,
    tb_requisition_item,
    tb_inventory,
    tb_inventory_count,
    tb_alert
RESTART IDENTITY CASCADE;
""")
cursor.execute("""
TRUNCATE TABLE
    tb_log_user,
    tb_log_product,
    tb_log_supplier,
    tb_log_stock_batch,
    tb_log_requisition,
    tb_log_inventory,
    tb_log_alert
RESTART IDENTITY;
""")
conn.commit()

print("Banco resetado com sucesso")

Banco resetado com sucesso


# DOMÍNIO: EMPRESAS DO GRUPO J&F E CATÁLOGO DE ALIMENTOS

In [39]:
# Dados fictícios ambientados nas empresas do grupo J&F: os nomes das
# subsidiárias são usados apenas para tematizar cozinhas, fornecedores e
# e-mails de usuários. CNPJs, e-mails e demais identificadores são gerados
# com Faker e não correspondem a dados reais.
SUBSIDIARIES = [
    {"code": "JBS", "name": "JBS S.A.", "domain": "jbs.com.br"},
    {"code": "SEARA", "name": "Seara Alimentos", "domain": "seara.com.br"},
    {"code": "FRIBOI", "name": "Friboi", "domain": "friboi.com.br"},
    {"code": "SWIFT", "name": "Swift", "domain": "swift.com.br"},
    {"code": "VIGOR", "name": "Vigor Alimentos", "domain": "vigoralimentos.com.br"},
    {"code": "FLORA", "name": "Flora", "domain": "flora.com.br"},
    {"code": "ORIGINAL", "name": "Banco Original", "domain": "bancooriginal.com.br"},
    {"code": "ELDORADO", "name": "Eldorado Brasil", "domain": "eldoradobrasil.com.br"},
    {"code": "PICPAY", "name": "PicPay", "domain": "picpay.com"},
    {"code": "3CORACOES", "name": "Três Corações Alimentos", "domain": "3coracoes.com.br"},
    {"code": "CANALRURAL", "name": "Canal Rural", "domain": "canalrural.com.br"},
    {"code": "JF", "name": "J&F Investimentos", "domain": "jf.com.br"},
]

# Fornecedores "verticalizados" do próprio grupo (produzem para as cozinhas)
JF_FOOD_SUPPLIERS = ["JBS", "SEARA", "FRIBOI", "SWIFT", "VIGOR", "FLORA", "3CORACOES"]

# Cidades brasileiras reais (nome, UF) usadas para localizar as cozinhas
# industriais — evita os topônimos sintéticos do Faker (ex.: "Alves/SC").
REAL_CITIES = [
    ("São Paulo", "SP"), ("Barueri", "SP"), ("Lins", "SP"), ("Barretos", "SP"),
    ("Andradina", "SP"), ("Bauru", "SP"), ("Marília", "SP"), ("Campinas", "SP"),
    ("Ribeirão Preto", "SP"), ("São José do Rio Preto", "SP"), ("Osasco", "SP"),
    ("Rio de Janeiro", "RJ"), ("Niterói", "RJ"), ("Duque de Caxias", "RJ"),
    ("Belo Horizonte", "MG"), ("Uberlândia", "MG"), ("Uberaba", "MG"),
    ("Juiz de Fora", "MG"), ("Contagem", "MG"), ("Montes Claros", "MG"),
    ("Curitiba", "PR"), ("Londrina", "PR"), ("Maringá", "PR"), ("Toledo", "PR"),
    ("Cascavel", "PR"), ("Ponta Grossa", "PR"),
    ("Porto Alegre", "RS"), ("Caxias do Sul", "RS"), ("Passo Fundo", "RS"),
    ("Santa Maria", "RS"), ("Pelotas", "RS"),
    ("Florianópolis", "SC"), ("Itajaí", "SC"), ("Forquilhinha", "SC"),
    ("Chapecó", "SC"), ("Joinville", "SC"), ("Blumenau", "SC"),
    ("Campo Grande", "MS"), ("Dourados", "MS"), ("Três Lagoas", "MS"),
    ("Cuiabá", "MT"), ("Rondonópolis", "MT"), ("Barra do Garças", "MT"),
    ("Sinop", "MT"), ("Várzea Grande", "MT"),
    ("Goiânia", "GO"), ("Rio Verde", "GO"), ("Anápolis", "GO"), ("Jataí", "GO"),
    ("Brasília", "DF"),
    ("Salvador", "BA"), ("Feira de Santana", "BA"), ("Ilhéus", "BA"),
    ("Barreiras", "BA"),
    ("Recife", "PE"), ("Caruaru", "PE"), ("Petrolina", "PE"),
    ("Fortaleza", "CE"), ("Juazeiro do Norte", "CE"),
    ("Natal", "RN"), ("João Pessoa", "PB"), ("Maceió", "AL"), ("Aracaju", "SE"),
    ("São Luís", "MA"), ("Imperatriz", "MA"), ("Teresina", "PI"),
    ("Belém", "PA"), ("Marabá", "PA"), ("Santarém", "PA"), ("Ananindeua", "PA"),
    ("Manaus", "AM"), ("Porto Velho", "RO"), ("Rio Branco", "AC"),
    ("Boa Vista", "RR"), ("Macapá", "AP"), ("Palmas", "TO"),
    ("Vitória", "ES"), ("Vila Velha", "ES"), ("Linhares", "ES"),
]

FOOD_GROUPS = {
    "beef": {
        "label": "Carnes Bovinas", "unit_kind": "weight",
        "brands": ["JBS", "Friboi", "Swift"],
        "items": [
            "Acém", "Alcatra", "Contrafilé", "Costela Bovina", "Cupim",
            "Fraldinha", "Filé Mignon", "Maminha", "Músculo", "Patinho",
            "Picanha", "Lagarto", "Coxão Mole", "Coxão Duro", "Peito Bovino",
            "Paleta Bovina", "Aba de Filé", "Bisteca Bovina", "Capa de Filé",
            "Ponta de Agulha",
        ],
    },
    "pork": {
        "label": "Carnes Suínas", "unit_kind": "weight",
        "brands": ["Seara", "JBS"],
        "items": [
            "Pernil Suíno", "Lombo Suíno", "Bisteca Suína", "Costela Suína",
            "Paleta Suína", "Bacon", "Linguiça Suína", "Toucinho",
            "Filé Suíno", "Copa Lombo", "Barriga Suína", "Pé de Porco",
        ],
    },
    "poultry": {
        "label": "Aves", "unit_kind": "weight",
        "brands": ["Seara"],
        "items": [
            "Peito de Frango", "Coxa de Frango", "Sobrecoxa", "Asa de Frango",
            "Frango Inteiro", "Filé de Peito de Frango", "Coração de Frango",
            "Frango a Passarinho", "Peru", "Chester",
        ],
    },
    "fish": {
        "label": "Peixes e Frutos do Mar", "unit_kind": "weight",
        "brands": ["Seara", "Diversos"],
        "items": [
            "Filé de Tilápia", "Salmão", "Camarão", "Bacalhau",
            "Filé de Merluza", "Lula", "Polvo", "Sardinha", "Atum", "Pescada",
        ],
    },
    "dairy": {
        "label": "Laticínios e Frios", "unit_kind": "volume",
        "brands": ["Vigor", "Leco", "Diversos"],
        "items": [
            "Leite Integral", "Leite Desnatado", "Queijo Mussarela",
            "Queijo Prato", "Requeijão", "Manteiga", "Margarina",
            "Iogurte Natural", "Presunto", "Mortadela", "Salame",
            "Creme de Leite", "Leite Condensado", "Queijo Parmesão", "Ricota",
        ],
    },
    "produce_veg": {
        "label": "Hortifruti - Legumes e Verduras", "unit_kind": "weight",
        "brands": ["Diversos"],
        "items": [
            "Tomate", "Cebola", "Batata", "Cenoura", "Alface", "Repolho",
            "Abobrinha", "Chuchu", "Pimentão", "Beterraba", "Vagem",
            "Brócolis", "Couve-Flor", "Pepino", "Mandioca", "Abóbora",
            "Espinafre", "Rúcula", "Alho", "Milho Verde",
        ],
    },
    "produce_fruit": {
        "label": "Hortifruti - Frutas", "unit_kind": "weight",
        "brands": ["Diversos"],
        "items": [
            "Banana", "Maçã", "Laranja", "Mamão", "Melancia", "Abacaxi",
            "Limão", "Manga", "Uva", "Melão", "Morango", "Abacate", "Pera",
            "Tangerina",
        ],
    },
    "grains": {
        "label": "Grãos, Cereais e Massas", "unit_kind": "weight",
        "brands": ["Diversos"],
        "items": [
            "Arroz Branco", "Feijão Carioca", "Feijão Preto",
            "Macarrão Espaguete", "Macarrão Parafuso", "Farinha de Trigo",
            "Farinha de Mandioca", "Fubá", "Aveia", "Lentilha",
            "Grão de Bico", "Quinoa", "Polenta", "Farinha de Rosca",
        ],
    },
    "bakery": {
        "label": "Panificação", "unit_kind": "count",
        "brands": ["Diversos"],
        "items": [
            "Pão Francês", "Pão de Forma", "Pão de Hambúrguer",
            "Pão de Hot Dog", "Torrada", "Biscoito Cream Cracker",
            "Biscoito Maisena", "Bolo Pronto", "Massa para Pastel",
            "Massa para Pizza",
        ],
    },
    "seasoning": {
        "label": "Temperos e Condimentos", "unit_kind": "weight",
        "brands": ["Diversos"],
        "items": [
            "Sal Refinado", "Açúcar Refinado", "Açúcar Cristal",
            "Pimenta do Reino", "Colorau", "Cominho", "Alho em Pó",
            "Cebola em Pó", "Orégano", "Louro", "Molho de Tomate", "Ketchup",
            "Mostarda", "Maionese", "Vinagre", "Caldo de Carne",
            "Caldo de Galinha", "Shoyu", "Azeite de Oliva",
        ],
    },
    "oils": {
        "label": "Óleos e Gorduras", "unit_kind": "volume",
        "brands": ["Diversos"],
        "items": [
            "Óleo de Soja", "Óleo de Girassol", "Óleo de Milho",
            "Banha Suína", "Gordura Vegetal Hidrogenada",
        ],
    },
    "beverages": {
        "label": "Bebidas", "unit_kind": "volume",
        "brands": ["Três Corações", "Diversos"],
        "items": [
            "Água Mineral", "Suco de Laranja", "Suco de Uva",
            "Refrigerante Cola", "Refrigerante Guaraná",
            "Café Torrado e Moído", "Achocolatado", "Chá Mate",
            "Suco Concentrado de Frutas", "Água de Coco",
        ],
    },
    "frozen": {
        "label": "Congelados", "unit_kind": "weight",
        "brands": ["Seara", "JBS", "Diversos"],
        "items": [
            "Batata Frita Congelada", "Hambúrguer Congelado",
            "Nuggets de Frango", "Empanado de Frango",
            "Legumes Congelados", "Polpa de Fruta Congelada",
            "Massa Folhada Congelada", "Pão de Queijo Congelado",
            "Almôndega Congelada", "Lasanha Congelada",
        ],
    },
    "cleaning": {
        "label": "Produtos de Limpeza", "unit_kind": "volume",
        "brands": ["Flora", "Diversos"],
        "items": [
            "Detergente Neutro", "Desinfetante", "Água Sanitária",
            "Sabão em Pó", "Álcool 70%", "Esponja de Aço",
            "Pano Multiuso", "Sabão em Barra", "Limpador Multiuso",
            "Removedor de Gordura", "Saco de Lixo", "Cloro Concentrado",
        ],
    },
    "disposables": {
        "label": "Descartáveis e Embalagens", "unit_kind": "count",
        "brands": ["Diversos"],
        "items": [
            "Copo Descartável", "Prato Descartável", "Guardanapo de Papel",
            "Papel Toalha", "Papel Alumínio", "Filme Plástico PVC",
            "Luva Descartável", "Touca Descartável",
            "Embalagem para Marmitex", "Palito de Dente",
            "Saco Plástico Transparente", "Papel Manteiga",
        ],
    },
}


def esc_len(text, limit):
    assert len(text) <= limit, f"{len(text)} > {limit}: {text!r}"
    return text


def collect_unique(generator_fn, n, key=lambda row: row[0]):
    """Consome um gerador (potencialmente infinito) até acumular n itens
    únicos pela chave informada. Usado para as tabelas de domínio, que têm
    colunas UNIQUE e por isso não podem receber duplicatas na hora do INSERT."""
    seen = set()
    out = []
    for row in generator_fn():
        k = key(row)
        if k in seen:
            continue
        seen.add(k)
        out.append(row)
        if len(out) >= n:
            break
    if len(out) < n:
        raise RuntimeError(f"Gerador produziu só {len(out)} linhas únicas de {n}.")
    return out


def slugify(text):
    nfkd = unicodedata.normalize("NFKD", text)
    ascii_text = nfkd.encode("ascii", "ignore").decode("ascii").lower()
    return re.sub(r"[^a-z0-9]+", ".", ascii_text).strip(".")


def unique_pairs(ids_a, ids_b, n):
    seen = set()
    out = []
    attempts = 0
    while len(out) < n and attempts < n * 100:
        pair = (random.choice(ids_a), random.choice(ids_b))
        attempts += 1
        if pair in seen:
            continue
        seen.add(pair)
        out.append(pair)
    if len(out) < n:
        raise RuntimeError(f"unique_pairs: só consegui {len(out)} pares únicos de {n}")
    return out


print("Domínio J&F e helpers carregados")

Domínio J&F e helpers carregados


# 1. PERFIS DE ACESSO (tb_profile)

In [40]:
ROLES = [
    ("COMPRADOR", "Comprador — responsável por comprar os produtos em falta no estoque"),
    ("ESTOQUISTA", "Estoquista — responsável por controlar o estoque"),
    ("SUPERVISOR", "Supervisor — supervisiona compradores e estoquistas"),
]
LEVELS = [("JR", "Júnior"), ("PL", "Pleno"), ("SR", "Sênior"), ("ESP", "Especialista")]
# Unidade de negócio dentro de cada subsidiária: só 3 papéis x 12
# subsidiárias x 4 níveis dariam 144 combinações, abaixo do mínimo de 500
# exigido para a tabela; essa dimensão extra fecha em 576 (3x12x4x4).
BUSINESS_UNITS = [1, 2, 3, 4]


def profile_combos():
    for (role_code, role_desc), sub, (lvl_code, lvl_desc), unit in itertools.product(
        ROLES, SUBSIDIARIES, LEVELS, BUSINESS_UNITS
    ):
        access_type = f"{role_code}_{sub['code']}_{lvl_code}_U{unit}"
        description = f"{role_desc} — {sub['name']} (Nível {lvl_desc}, Unidade {unit})"
        yield access_type, description


def profile_fallback():
    for i in itertools.count(1):
        yield f"PERFIL_GENERICO_{i:04d}", f"Perfil de acesso genérico {i}"


profiles = collect_unique(lambda: itertools.chain(profile_combos(), profile_fallback()), 576)

rows = [(esc_len(access_type, 50), description) for access_type, description in profiles]
result = execute_values(
    cursor,
    "INSERT INTO tb_profile (access_type, description) VALUES %s RETURNING id_profile",
    rows,
    fetch=True,
    page_size=1000,
)
profile_ids = [r[0] for r in result]

conn.commit()
print(f"{len(profile_ids)} perfis de acesso criados")

576 perfis de acesso criados


# 2. COZINHAS INDUSTRIAIS (tb_kitchen)

In [41]:
def kitchen_combos():
    for sub in SUBSIDIARIES:
        for n in range(1, 60):
            city, state = random.choice(REAL_CITIES)
            name = f"Cozinha Industrial {sub['name']} - {city}/{state} (Unidade {n:02d})"
            yield f"{sub['code']}-{n:04d}", name, sub["domain"]


def kitchen_fallback():
    for i in itertools.count(1):
        city, state = random.choice(REAL_CITIES)
        yield f"GEN-{i:05d}", f"Cozinha Industrial Genérica {i}", random.choice(SUBSIDIARIES)["domain"]


kitchens_raw = collect_unique(lambda: itertools.chain(kitchen_combos(), kitchen_fallback()), 552)

rows = []
domains = []
for code, name, domain in kitchens_raw:
    city, state = random.choice(REAL_CITIES)
    address = f"{fake.street_address()}, {city}/{state}"[:255]
    active = random.random() < 0.93
    created_at = fake.date_time_between(start_date="-5y", end_date="-1y")
    rows.append((esc_len(name, 120), esc_len(code, 20), address, active, created_at))
    domains.append(domain)

result = execute_values(
    cursor,
    "INSERT INTO tb_kitchen (name, code, address, active, created_at) VALUES %s RETURNING id_kitchen",
    rows,
    fetch=True,
    page_size=1000,
)
kitchen_ids = [r[0] for r in result]
kitchen_domain_by_id = dict(zip(kitchen_ids, domains))

conn.commit()
print(f"{len(kitchen_ids)} cozinhas industriais criadas (grupo J&F)")

552 cozinhas industriais criadas (grupo J&F)


# 3. CATEGORIAS DE PRODUTO (tb_category)

In [42]:
CATEGORY_VARIANTS = ["", " (Embalagem Institucional)", " (Embalagem Fracionada)"]


def category_combos():
    for group_key, group in FOOD_GROUPS.items():
        for item in group["items"]:
            for variant in CATEGORY_VARIANTS:
                name = f"{group['label']} - {item}{variant}"
                if len(name) > 80:
                    continue
                yield name, f"Itens de {item.lower()} para cozinhas industriais do grupo J&F.", group_key


def category_fallback():
    for i in itertools.count(1):
        yield f"Categoria Geral {i}", f"Categoria genérica {i}", None


categories_raw = collect_unique(lambda: itertools.chain(category_combos(), category_fallback()), 560)

rows = [(esc_len(name, 80), esc_len(description, 255)) for name, description, _ in categories_raw]
group_keys = [group_key for _, _, group_key in categories_raw]

result = execute_values(
    cursor,
    "INSERT INTO tb_category (name, description) VALUES %s RETURNING id_category",
    rows,
    fetch=True,
    page_size=1000,
)
category_ids = [r[0] for r in result]

cat_ids_by_group = {}
generic_category_ids = []
for id_category, group_key in zip(category_ids, group_keys):
    if group_key is None:
        generic_category_ids.append(id_category)
    else:
        cat_ids_by_group.setdefault(group_key, []).append(id_category)

# categorias "fallback" (sem grupo definido) entram no pool de todos os grupos
for group_key in FOOD_GROUPS:
    cat_ids_by_group.setdefault(group_key, []).extend(generic_category_ids)

conn.commit()
print(f"{len(category_ids)} categorias de produto criadas")

560 categorias de produto criadas


# 4. UNIDADES DE MEDIDA (tb_measurement_unit)

In [43]:
def humanize_size(size):
    m = re.match(r"(\d+)(G|KG|ML|L|UN)$", size)
    if not m:
        return size
    qty, unit = m.groups()
    return f"{qty}{ {'G': 'g', 'KG': 'kg', 'ML': 'ml', 'L': 'L', 'UN': 'un'}[unit] }"


CONTAINERS_WEIGHT = [("CX", "Caixa"), ("PCT", "Pacote"), ("SC", "Saco"), ("BD", "Balde"), ("BB", "Big Bag")]
CONTAINERS_VOLUME = [("GL", "Galão"), ("LT", "Lata"), ("GRF", "Garrafa"), ("POT", "Pote"), ("BDJ", "Bandeja")]
CONTAINERS_COUNT = [("CX", "Caixa"), ("PCT", "Pacote"), ("DZ", "Dúzia"), ("ENG", "Engradado"), ("FD", "Fardo")]

WEIGHT_SIZES = [
    "50G", "100G", "150G", "200G", "250G", "300G", "400G", "500G", "600G",
    "750G", "900G", "1KG", "2KG", "3KG", "4KG", "5KG", "8KG", "10KG",
    "15KG", "20KG", "25KG", "30KG", "40KG", "50KG",
]
VOLUME_SIZES = [
    "100ML", "150ML", "200ML", "250ML", "300ML", "350ML", "500ML",
    "600ML", "750ML", "900ML", "1L", "2L", "3L", "5L", "10L", "15L", "20L", "50L", "200L",
]
COUNT_SIZES = [
    "3UN", "6UN", "10UN", "12UN", "20UN", "24UN", "30UN", "40UN", "50UN",
    "60UN", "100UN", "144UN", "200UN", "250UN", "500UN",
]
BARE_UNITS = [
    ("KG", "Quilograma", "weight"), ("G", "Grama", "weight"),
    ("TON", "Tonelada", "weight"), ("L", "Litro", "volume"),
    ("ML", "Mililitro", "volume"), ("UN", "Unidade", "count"),
    ("DZ", "Dúzia", "count"), ("PAR", "Par", "count"), ("M", "Metro", "count"),
]


def unit_simple_combos():
    for ccode, cname in CONTAINERS_WEIGHT:
        for size in WEIGHT_SIZES:
            symbol = f"{ccode}{size}"
            if len(symbol) <= 10:
                yield symbol, f"{cname} de {humanize_size(size)}", "weight"
    for ccode, cname in CONTAINERS_VOLUME:
        for size in VOLUME_SIZES:
            symbol = f"{ccode}{size}"
            if len(symbol) <= 10:
                yield symbol, f"{cname} de {humanize_size(size)}", "volume"
    for ccode, cname in CONTAINERS_COUNT:
        for size in COUNT_SIZES:
            symbol = f"{ccode}{size}"
            if len(symbol) <= 10:
                yield symbol, f"{cname} com {humanize_size(size)}", "count"


def unit_multipack_combos():
    for ccode, cname in [("FD", "Fardo"), ("CX", "Caixa"), ("ENG", "Engradado")]:
        for mult in [3, 6, 10, 12, 20, 24, 30, 50]:
            for base in WEIGHT_SIZES[:10] + VOLUME_SIZES[:10]:
                symbol = f"{ccode}{mult}X{base}"
                if len(symbol) <= 10:
                    kind = "volume" if "ML" in base or base.endswith("L") else "weight"
                    yield symbol, f"{cname} com {mult} unidades de {humanize_size(base)}"[:60], kind


def unit_fallback():
    for i in itertools.count(1):
        yield f"UN{i:04d}"[:10], f"Unidade genérica {i}", "count"


units_raw = collect_unique(
    lambda: itertools.chain(BARE_UNITS, unit_simple_combos(), unit_multipack_combos(), unit_fallback()),
    560,
)

rows = [(esc_len(symbol, 10), esc_len(description, 60)) for symbol, description, _ in units_raw]
kinds = [kind for _, _, kind in units_raw]

result = execute_values(
    cursor,
    "INSERT INTO tb_measurement_unit (symbol, description) VALUES %s RETURNING id_unit",
    rows,
    fetch=True,
    page_size=1000,
)

unit_ids_by_kind = {}
for id_unit, kind in zip((r[0] for r in result), kinds):
    unit_ids_by_kind.setdefault(kind, []).append(id_unit)

conn.commit()
print(f"{len(result)} unidades de medida criadas")

560 unidades de medida criadas


# 5. FORNECEDORES (tb_supplier)

In [44]:
SECTOR_SUFFIXES = [
    "Distribuidora de Alimentos", "Atacado de Hortifruti", "Frigorífico",
    "Laticínios", "Embalagens e Descartáveis", "Produtos de Limpeza",
    "Bebidas e Insumos", "Comércio de Cereais", "Importadora de Alimentos",
    "Logística e Distribuição",
]


def supplier_combos():
    for sub_code in JF_FOOD_SUPPLIERS:
        sub = next(s for s in SUBSIDIARIES if s["code"] == sub_code)
        yield f"{sub['name']} Distribuição Interna LTDA", sub["domain"]
    while True:
        legal_name = f"{fake.company()} {random.choice(SECTOR_SUFFIXES)}"[:150]
        yield legal_name, None


supplier_meta = []
for legal_name, domain_hint in supplier_combos():
    if len(supplier_meta) >= 550:
        break
    supplier_meta.append((legal_name, domain_hint))

rows = []
used_cnpj = set()
for legal_name, domain_hint in supplier_meta:
    cnpj = fake.cnpj()
    while cnpj in used_cnpj:
        cnpj = fake.cnpj()
    used_cnpj.add(cnpj)

    local = slugify(legal_name.split(" ")[0]) or "contato"
    domain = domain_hint or f"{local}.com.br"
    email = f"contato.{local}@{domain}"[:150]
    whatsapp = f"+55 {random.randint(11, 99)} 9{random.randint(1000, 9999)}-{random.randint(1000, 9999)}"
    rating = random.choices([1, 2, 3, 4, 5], weights=[3, 7, 20, 40, 30])[0]
    active = random.random() < 0.92
    created_at = fake.date_time_between(start_date="-5y", end_date="-6m")

    rows.append((esc_len(legal_name, 150), cnpj, email, whatsapp[:20], rating, active, created_at))

result = execute_values(
    cursor,
    """INSERT INTO tb_supplier (legal_name, cnpj, email, whatsapp, rating, active, created_at)
       VALUES %s RETURNING id_supplier""",
    rows,
    fetch=True,
    page_size=1000,
)
supplier_ids = [r[0] for r in result]
product_supplier_map = {}  # preenchido na seção 8

conn.commit()
print(f"{len(supplier_ids)} fornecedores criados (incluindo empresas do grupo J&F)")

550 fornecedores criados (incluindo empresas do grupo J&F)


# 6. PRODUTOS (tb_product)

In [45]:
SIZE_VARIANTS = ["Embalagem Padrão", "Embalagem Institucional", "Fracionado"]


def product_combos():
    for group_key, group in FOOD_GROUPS.items():
        for item in group["items"]:
            for brand in group["brands"]:
                for variant in SIZE_VARIANTS:
                    name = f"{item} {brand} - {variant}"
                    if len(name) > 150:
                        continue
                    yield name, brand, group_key


def product_fallback():
    for i in itertools.count(1):
        group_key = random.choice(list(FOOD_GROUPS.keys()))
        yield f"Produto Genérico {i}", "Diversos", group_key


products_raw = collect_unique(lambda: itertools.chain(product_combos(), product_fallback()), 560)

rows = []
used_barcodes = set()
for name, brand, group_key in products_raw:
    barcode = fake.ean13()
    while barcode in used_barcodes:
        barcode = fake.ean13()
    used_barcodes.add(barcode)

    unit_kind = FOOD_GROUPS[group_key]["unit_kind"]
    id_category = random.choice(cat_ids_by_group[group_key])
    id_unit = random.choice(unit_ids_by_kind[unit_kind])
    active = random.random() < 0.95
    created_at = fake.date_time_between(start_date="-4y", end_date="-3m")

    rows.append((esc_len(name, 150), esc_len(brand, 80), id_category, id_unit, barcode, active, created_at))

result = execute_values(
    cursor,
    """INSERT INTO tb_product (name, brand, id_category, id_unit, barcode, active, created_at)
       VALUES %s RETURNING id_product""",
    rows,
    fetch=True,
    page_size=1000,
)
product_ids = [r[0] for r in result]

conn.commit()
print(f"{len(product_ids)} produtos criados")

560 produtos criados


# 7. USUÁRIOS (tb_user)

In [46]:
ROLE_LABELS = [
    "Cozinheiro", "Auxiliar de Cozinha", "Gestor de Estoque", "Comprador",
    "Auditor Interno", "Nutricionista", "Supervisor de Cozinha",
    "Analista de TI", "Gerente Regional", "Assistente Administrativo",
]

rows = []
user_uuids = []
used_emails = set()
for _ in range(550):
    name = fake.name()
    has_kitchen = random.random() < 0.85
    id_kitchen = random.choice(kitchen_ids) if has_kitchen else None
    domain = kitchen_domain_by_id[id_kitchen] if id_kitchen else random.choice(SUBSIDIARIES)["domain"]

    local = slugify(name)
    email = f"{local}@{domain}"
    suffix = 1
    while email in used_emails:
        suffix += 1
        email = f"{local}{suffix}@{domain}"
    used_emails.add(email)

    password_hash = "$2b$12$" + "".join(
        random.choices("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789./", k=53)
    )
    created_at = fake.date_time_between(start_date="-4y", end_date="-1m")
    last_login = fake.date_time_between(start_date=created_at, end_date="now") if random.random() < 0.85 else None
    id_user = fake.uuid4()

    rows.append((
        id_user, esc_len(name, 120), email[:150], password_hash,
        random.choice(ROLE_LABELS), id_kitchen, random.choice(profile_ids),
        random.random() < 0.92, last_login, created_at,
    ))
    user_uuids.append(id_user)

execute_values(
    cursor,
    """INSERT INTO tb_user
        (id_user, name, email, password_hash, role, id_kitchen, id_profile, active, last_login, created_at)
       VALUES %s""",
    rows,
    page_size=1000,
)

conn.commit()
print(f"{len(user_uuids)} usuários criados")

550 usuários criados


# 8. PRODUTO x FORNECEDOR (tb_product_supplier)

In [47]:
product_supplier_pairs = unique_pairs(product_ids, supplier_ids, 550)

rows = []
for id_product, id_supplier in product_supplier_pairs:
    rows.append((
        id_product, id_supplier, f"SUP{id_supplier:04d}-PRD{id_product:04d}",
        round(random.uniform(2, 800), 2), random.randint(1, 30),
    ))
    product_supplier_map.setdefault(id_product, []).append(id_supplier)

execute_values(
    cursor,
    """INSERT INTO tb_product_supplier
        (id_product, id_supplier, supplier_code, reference_price, lead_time_days)
       VALUES %s""",
    rows,
    page_size=1000,
)

conn.commit()
print(f"{len(product_supplier_pairs)} vínculos produto x fornecedor criados")

550 vínculos produto x fornecedor criados


# 9. PARÂMETROS DE ESTOQUE POR COZINHA (tb_product_kitchen_parameter)

In [48]:
product_kitchen_pairs = unique_pairs(product_ids, kitchen_ids, 550)

rows = []
for id_product, id_kitchen in product_kitchen_pairs:
    min_stock = round(random.uniform(5, 300), 3)
    max_stock = round(min_stock * random.uniform(1.5, 4), 3)
    avg_consumption = round(min_stock / random.uniform(2, 6), 3)
    rows.append((id_product, id_kitchen, min_stock, max_stock, avg_consumption))

execute_values(
    cursor,
    """INSERT INTO tb_product_kitchen_parameter
        (id_product, id_kitchen, min_stock, max_stock, average_daily_consumption)
       VALUES %s""",
    rows,
    page_size=1000,
)

conn.commit()
print(f"{len(product_kitchen_pairs)} parâmetros de estoque por cozinha criados")

550 parâmetros de estoque por cozinha criados


# 10. LOTES DE ESTOQUE (tb_stock_batch)

In [49]:
rows = []
for i in range(550):
    id_product = random.choice(product_ids)
    id_kitchen = random.choice(kitchen_ids)
    candidates = product_supplier_map.get(id_product)
    if candidates and random.random() < 0.6:
        id_supplier = random.choice(candidates)
    elif random.random() < 0.9:
        id_supplier = random.choice(supplier_ids)
    else:
        id_supplier = None

    entry_date = fake.date_time_between(start_date="-2y", end_date="-1d").date()
    initial_quantity = round(random.uniform(5, 1500), 3)
    current_quantity = round(initial_quantity * (1 - random.uniform(0, 1)), 3)

    roll = random.random()
    if roll < 0.08:
        expiration_date = None
    elif roll < 0.25:
        expiration_date = entry_date + timedelta(days=random.randint(1, 60))
    else:
        expiration_date = entry_date + timedelta(days=random.randint(15, 365))

    status = random.choices(["ACTIVE", "WRITTEN_OFF", "EXPIRED", "CANCELLED"], weights=[70, 12, 10, 8])[0]

    rows.append((
        id_product, id_kitchen, id_supplier, f"{entry_date:%Y%m}-{i + 1:05d}",
        f"NF-{random.randint(100000, 999999)}", initial_quantity, current_quantity,
        entry_date, expiration_date, round(random.uniform(2, 800), 2), status,
    ))

result = execute_values(
    cursor,
    """INSERT INTO tb_stock_batch
        (id_product, id_kitchen, id_supplier, batch_number, invoice_number,
         initial_quantity, current_quantity, entry_date, expiration_date, unit_price, status)
       VALUES %s RETURNING id_batch""",
    rows,
    fetch=True,
    page_size=1000,
)
batch_ids = [r[0] for r in result]

conn.commit()
print(f"{len(batch_ids)} lotes de estoque criados")

550 lotes de estoque criados


# 11. REQUISIÇÕES (tb_requisition)

In [50]:
ORIGINS = ["KITCHEN", "MOBILE_APP", "SYSTEM", "MANUAL", "ERP_INTEGRATION"]
REASONS = [
    "Reposição de estoque mínimo", "Solicitação da cozinha", "Substituição de lote vencido",
    "Ajuste de inventário", "Pedido programado mensal", None, None,
]

rows = []
for _ in range(550):
    created_at = fake.date_time_between(start_date="-2y", end_date="now")
    status = random.choices(["UNDER_REVIEW", "APPROVED", "REJECTED", "CANCELLED"], weights=[30, 45, 15, 10])[0]
    if status in ("APPROVED", "REJECTED"):
        approved_at = created_at + timedelta(hours=random.randint(1, 72))
        id_approver_user = random.choice(user_uuids)
    else:
        approved_at = None
        id_approver_user = None

    rows.append((
        random.choices(["PURCHASE", "TRANSFER", "CONSUMPTION"], weights=[50, 25, 25])[0],
        random.choice(ORIGINS), status, random.choice(REASONS), random.choice(kitchen_ids),
        random.choice(user_uuids), id_approver_user, created_at, approved_at,
    ))

result = execute_values(
    cursor,
    """INSERT INTO tb_requisition
        (requisition_type, origin, status, reason, id_kitchen,
         id_requester_user, id_approver_user, created_at, approved_at)
       VALUES %s RETURNING id_requisition""",
    rows,
    fetch=True,
    page_size=1000,
)
requisition_ids = [r[0] for r in result]

conn.commit()
print(f"{len(requisition_ids)} requisições criadas")

550 requisições criadas


# 12. ITENS DE REQUISIÇÃO (tb_requisition_item)

In [51]:
rows = []
for _ in range(550):
    id_suggested_supplier = random.choice(supplier_ids) if random.random() < 0.7 else None
    estimated_price = round(random.uniform(2, 500), 2) if random.random() < 0.85 else None
    note = fake.sentence(nb_words=6) if random.random() < 0.3 else None

    rows.append((
        random.choice(requisition_ids), random.choice(product_ids), id_suggested_supplier,
        round(random.uniform(1, 200), 3), estimated_price, note,
    ))

execute_values(
    cursor,
    """INSERT INTO tb_requisition_item
        (id_requisition, id_product, id_suggested_supplier, quantity, estimated_price, note)
       VALUES %s""",
    rows,
    page_size=1000,
)

conn.commit()
print(f"{len(rows)} itens de requisição criados")

550 itens de requisição criados


# 13. INVENTÁRIOS (tb_inventory)

In [52]:
# Notas condizentes com o status: uma contagem OPEN não teria uma nota de
# "finalizado sem divergências", por exemplo — cada status tem seu próprio
# conjunto de motivos plausíveis.
INVENTORY_NOTE_TEMPLATES = {
    "OPEN": [
        "Contagem em andamento, aguardando conferência do turno seguinte.",
        "Inventário cíclico mensal em andamento.",
        "Conferência física iniciada para os itens perecíveis.",
        "Contagem em andamento devido a suspeita de divergência no estoque.",
    ],
    "CLOSED": [
        "Contagem finalizada sem divergências relevantes.",
        "Inventário concluído após conferência física completa.",
        "Fechado dentro do prazo previsto, sem pendências.",
        "Contagem rotineira concluída; pequenos ajustes registrados.",
    ],
    "CANCELLED": [
        "Cancelado por indisponibilidade da equipe responsável.",
        "Interrompido devido a falha no sistema durante a contagem.",
        "Cancelado para nova contagem após divergência crítica identificada.",
        "Suspenso a pedido do supervisor da cozinha.",
    ],
}

rows = []
for _ in range(550):
    started_at = fake.date_time_between(start_date="-2y", end_date="now")
    status = random.choices(["OPEN", "CLOSED", "CANCELLED"], weights=[20, 70, 10])[0]
    closed_at = started_at + timedelta(hours=random.randint(1, 48)) if status == "CLOSED" else None
    note = random.choice(INVENTORY_NOTE_TEMPLATES[status]) if random.random() < 0.25 else None

    rows.append((random.choice(kitchen_ids), random.choice(user_uuids), started_at, closed_at, status, note))

result = execute_values(
    cursor,
    """INSERT INTO tb_inventory (id_kitchen, id_responsible_user, started_at, closed_at, status, note)
       VALUES %s RETURNING id_inventory""",
    rows,
    fetch=True,
    page_size=1000,
)
inventory_ids = [r[0] for r in result]

conn.commit()
print(f"{len(inventory_ids)} inventários criados")

550 inventários criados


# 14. CONTAGENS DE INVENTÁRIO (tb_inventory_count)

In [53]:
# A coluna "divergence" não entra no INSERT: a trigger trg_calculate_divergence
# recalcula (physical_quantity - registered_quantity) automaticamente.
rows = []
for _ in range(550):
    registered_quantity = round(random.uniform(1, 1000), 3)
    physical_quantity = max(0, round(registered_quantity * (1 + random.uniform(-0.1, 0.1)), 3))
    note = fake.sentence(nb_words=6) if random.random() < 0.2 else None

    rows.append((random.choice(inventory_ids), random.choice(batch_ids), registered_quantity, physical_quantity, note))

execute_values(
    cursor,
    """INSERT INTO tb_inventory_count (id_inventory, id_batch, registered_quantity, physical_quantity, note)
       VALUES %s""",
    rows,
    page_size=1000,
)

conn.commit()
print(f"{len(rows)} contagens de inventário criadas")

550 contagens de inventário criadas


# 15. ALERTAS (tb_alert)

In [54]:
ALERT_TEMPLATES = {
    "STOCK": "Produto com estoque abaixo do mínimo configurado.",
    "EXPIRATION": "Lote próximo da data de validade ou vencido.",
    "QUALITY": "Divergência de qualidade reportada pela cozinha.",
    "MAINTENANCE": "Equipamento de refrigeração requer manutenção preventiva.",
    "TEMPERATURE": "Variação de temperatura detectada no armazenamento.",
    "OTHER": "Ocorrência registrada para acompanhamento.",
}

rows = []
for _ in range(550):
    alert_type = random.choices(list(ALERT_TEMPLATES.keys()), weights=[35, 30, 15, 10, 5, 5])[0]
    id_batch = random.choice(batch_ids) if random.random() < 0.7 else None
    id_product = random.choice(product_ids) if random.random() < 0.8 else None

    rows.append((
        alert_type, random.choices(["LOW", "MEDIUM", "HIGH", "CRITICAL"], weights=[25, 35, 25, 15])[0],
        id_batch, id_product, random.choice(kitchen_ids), ALERT_TEMPLATES[alert_type],
        random.random() < 0.4, fake.date_time_between(start_date="-2y", end_date="now"),
    ))

execute_values(
    cursor,
    """INSERT INTO tb_alert (type, severity, id_batch, id_product, id_kitchen, message, is_read, created_at)
       VALUES %s""",
    rows,
    page_size=1000,
)

conn.commit()
print(f"{len(rows)} alertas criados")

550 alertas criados


# RELATÓRIO EXECUTIVO J&F

In [55]:
print("\n" + "=" * 60)
print("     RELATÓRIO EXECUTIVO - INVENTRA (GRUPO J&F)")
print("=" * 60)

cursor.execute("""
    SELECT status, COUNT(*) FROM tb_stock_batch GROUP BY status ORDER BY COUNT(*) DESC
""")
print("\nLOTES DE ESTOQUE POR STATUS:")
for status, qtd in cursor.fetchall():
    print(f"   {status:<15} {qtd}")

cursor.execute("""
    SELECT type, severity, COUNT(*) FROM tb_alert GROUP BY type, severity ORDER BY COUNT(*) DESC LIMIT 10
""")
print("\nTOP 10 COMBINAÇÕES TIPO/SEVERIDADE DE ALERTA:")
for tipo, severidade, qtd in cursor.fetchall():
    print(f"   {tipo:<12} {severidade:<10} {qtd}")

cursor.execute("""
    SELECT k.name, COUNT(sb.id_batch) AS lotes
    FROM tb_kitchen k
    JOIN tb_stock_batch sb ON sb.id_kitchen = k.id_kitchen
    GROUP BY k.id_kitchen, k.name
    ORDER BY lotes DESC
    LIMIT 5
""")
print("\nTOP 5 COZINHAS POR VOLUME DE LOTES:")
for nome, lotes in cursor.fetchall():
    print(f"   {nome:<60} {lotes}")

tables = [
    "tb_profile", "tb_kitchen", "tb_category", "tb_measurement_unit", "tb_supplier",
    "tb_product", "tb_user", "tb_product_supplier", "tb_product_kitchen_parameter",
    "tb_stock_batch", "tb_requisition", "tb_requisition_item", "tb_inventory",
    "tb_inventory_count", "tb_alert",
]
log_tables = [
    "tb_log_user", "tb_log_product", "tb_log_supplier", "tb_log_stock_batch",
    "tb_log_requisition", "tb_log_inventory", "tb_log_alert",
]

print("\n" + "=" * 60)
print("     RESUMO FINAL - LINHAS POR TABELA")
print("=" * 60)
total_geral = 0
for table in tables:
    cursor.execute(f"SELECT COUNT(*) FROM {table}")
    qtd = cursor.fetchone()[0]
    total_geral += qtd
    flag = "OK" if qtd >= 500 else "FALTA"
    print(f"   {table:<32} {qtd:>6}  [{flag}]")

print("\n   Tabelas de log (preenchidas via trigger, sem INSERT direto):")
for table in log_tables:
    cursor.execute(f"SELECT COUNT(*) FROM {table}")
    print(f"   {table:<32} {cursor.fetchone()[0]:>6}")

print(f"\nTotal de linhas nas 15 tabelas de negócio: {total_geral}")
print("=" * 60)


     RELATÓRIO EXECUTIVO - INVENTRA (GRUPO J&F)

LOTES DE ESTOQUE POR STATUS:
   ACTIVE          395
   WRITTEN_OFF     68
   EXPIRED         45
   CANCELLED       42

TOP 10 COMBINAÇÕES TIPO/SEVERIDADE DE ALERTA:
   EXPIRATION   CRITICAL   430
   STOCK        HIGH       55
   STOCK        MEDIUM     54
   EXPIRATION   MEDIUM     50
   STOCK        LOW        48
   EXPIRATION   LOW        45
   EXPIRATION   HIGH       42
   STOCK        CRITICAL   35
   QUALITY      HIGH       25
   QUALITY      MEDIUM     22

TOP 5 COZINHAS POR VOLUME DE LOTES:
   Cozinha Industrial Swift - Uberaba/MG (Unidade 36)           6
   Cozinha Industrial Três Corações Alimentos - Juiz de Fora/MG (Unidade 03) 5
   Cozinha Industrial Seara Alimentos - Rio de Janeiro/RJ (Unidade 54) 5
   Cozinha Industrial Vigor Alimentos - Campinas/SP (Unidade 42) 5
   Cozinha Industrial Vigor Alimentos - Barueri/SP (Unidade 15) 5

     RESUMO FINAL - LINHAS POR TABELA
   tb_profile                          576  [OK]
   tb_ki

# FINALIZAR

In [56]:
cursor.close()
conn.close()

print("\nSeed Inventra (J&F) concluído com sucesso!")
print("Banco pronto para o sistema de gestão de estoque e cozinhas industriais")
print("\nDestaques da carga:")
print("   • 12 subsidiárias do grupo J&F usadas para tematizar cozinhas, fornecedores e usuários")
print("   • Cozinhas industriais localizadas em cidades reais do Brasil")
print("   • Catálogo de produtos alimentícios organizado por grupo (carnes, laticínios, hortifruti etc.)")
print("   • Fornecedores mesclando empresas verticalizadas do grupo e distribuidoras terceiras")
print("   • Lotes de estoque, requisições, inventários e alertas com volume >= 500 linhas cada")
print("   • Inserts em lote via execute_values (poucas viagens de rede por tabela, não uma por linha)")
print("   • Tabelas tb_log_* preenchidas automaticamente pelas triggers de auditoria")


Seed Inventra (J&F) concluído com sucesso!
Banco pronto para o sistema de gestão de estoque e cozinhas industriais

Destaques da carga:
   • 12 subsidiárias do grupo J&F usadas para tematizar cozinhas, fornecedores e usuários
   • Cozinhas industriais localizadas em cidades reais do Brasil
   • Catálogo de produtos alimentícios organizado por grupo (carnes, laticínios, hortifruti etc.)
   • Fornecedores mesclando empresas verticalizadas do grupo e distribuidoras terceiras
   • Lotes de estoque, requisições, inventários e alertas com volume >= 500 linhas cada
   • Inserts em lote via execute_values (poucas viagens de rede por tabela, não uma por linha)
   • Tabelas tb_log_* preenchidas automaticamente pelas triggers de auditoria


# ROLLBACK (OPCIONAL) — APAGAR DADOS DO SEED

In [ ]:
# Célula independente: não depende de nenhuma célula acima ter rodado nesta
# sessão do kernel. Roda ela sozinha sempre que quiser desfazer a carga
# deste seed — abre e fecha sua própria conexão (a célula FINALIZAR já
# fechou a conexão usada pelo resto do notebook).
#
# ATENÇÃO: destrutivo. Apaga TODAS as linhas das tabelas abaixo, não só as
# inseridas por este seed. RESTART IDENTITY zera os contadores de novo.
import os
import psycopg2
from dotenv import load_dotenv

load_dotenv()

rollback_conn = psycopg2.connect(
    host=os.getenv("DB_HOST"),
    port=os.getenv("DB_PORT"),
    database=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    sslmode=os.getenv("DB_SSLMODE", "require"),
)
rollback_cursor = rollback_conn.cursor()

rollback_cursor.execute("""
TRUNCATE TABLE
    tb_profile,
    tb_kitchen,
    tb_category,
    tb_measurement_unit,
    tb_user,
    tb_product,
    tb_supplier,
    tb_product_supplier,
    tb_product_kitchen_parameter,
    tb_stock_batch,
    tb_requisition,
    tb_requisition_item,
    tb_inventory,
    tb_inventory_count,
    tb_alert
RESTART IDENTITY CASCADE;
""")
rollback_cursor.execute("""
TRUNCATE TABLE
    tb_log_user,
    tb_log_product,
    tb_log_supplier,
    tb_log_stock_batch,
    tb_log_requisition,
    tb_log_inventory,
    tb_log_alert
RESTART IDENTITY;
""")
rollback_conn.commit()

rollback_cursor.close()
rollback_conn.close()

print("Rollback concluído: todos os dados inseridos pelo seed foram removidos.")